In [17]:
import requests
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
API_TOKEN = os.getenv("PRODUCT_HUNT_TOKEN", "")

In [ ]:
def get_trending_ai_products(API_TOKEN, min_votes=20):
    url = "https://api.producthunt.com/v2/api/graphql"

    ai_keywords = [
        "ai", "machine learning", "gpt", "llm", "neural",
        "deep learning", "model", "copilot", "assistant"
    ]

    one_week_ago = (datetime.now(UTC) - timedelta(days=7)).isoformat()

    query = f"""
    {{
      posts(first: 50, postedAfter: "{one_week_ago}") {{
        edges {{
          node {{
            name
            tagline
            url
            votesCount
            createdAt
            description
          }}
        }}
      }}
    }}
    """

    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, json={"query": query}, headers=headers)

    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return []

    data = response.json()

    filtered_products = []

    for edge in data.get("data", {}).get("posts", {}).get("edges", []):
        product = edge["node"]
        text = (product["name"] + " " + product["tagline"]).lower()

        if any(keyword in text for keyword in ai_keywords):
            if product["votesCount"] >= min_votes:
                filtered_products.append(product)

    # Sort by popularity
    filtered_products = sorted(
        filtered_products,
        key=lambda x: x["votesCount"],
        reverse=True
    )

    product_info = []

    for product in filtered_products:
        product_info.append({
            "Name": product["name"],
            "Tagline": product["tagline"],
            "Description": product["description"] or "No description",
            "Votes": product["votesCount"],
            "Link": product["url"]
        })

    return product_info

C:\Users\bsais\AppData\Local\Temp\ipykernel_3024\3524619633.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  one_week_ago = (datetime.utcnow() - timedelta(days=7)).isoformat()


In [21]:
product_info

[{'Name': 'Google Vids 2.0',
  'Tagline': 'Create, edit and share videos at no cost w/ new AI features',
  'Description': 'Create on-brand videos at scale with Google Vids, powered by Lyria 3 and Veo 3.1. Generate high-quality videos for free, add custom music, use avatars, and streamline workflows with Gemini across Workspace apps, all with powerful new AI capabilities.',
  'Link': 'https://www.producthunt.com/products/google?utm_campaign=producthunt-api&utm_medium=api-v2&utm_source=Application%3A+AI+News+letter+%28ID%3A+280564%29'},
 {'Name': 'OpenRouter Model Fusion',
  'Tagline': 'Run many models side by side and fuse the best answer',
  'Description': 'Model Fusion is a new public experiment from OpenRouter Labs. It runs your prompt through multiple models, analyzes their outputs, and uses a customizable "judge" model to fuse the best aspects into a single, superior response.',
  'Link': 'https://www.producthunt.com/products/openrouter?utm_campaign=producthunt-api&utm_medium=api-v